# Build Modelo · LAP01 multitracking multicámara

Notebook **padre** del build. El código de cada etapa está en el paquete **`lap01/`** (un `.py` por etapa) y este notebook lo llama en orden. `LAP01_Multitracking.ipynb` (carpeta de arriba) queda solo como guía.

```
CCTV → Tracking local (LAP01) → Re-ID multicámara → Mapa 2D del piso → trajectory_points
```

| Módulo `lap01/` | Qué contiene |
|---|---|
| `datos.py` | Metadatos y área útil de los videos. |
| `deteccion_apariencia.py` | `DetectorPersonas` (YOLO26m), firma de color, `ExtractorResNet18`, `Appearance`, `ClasificadorGeneroCLIP`, `recortar`. |
| `tracking_local.py` | Geometría, estados de un track y `ColorOcclusionTracker` (`local_id`). |
| `camaras_reid.py` | Homografías, `Camara`, encoder Re-ID (`ReIDPersonas`) y `Tracklet`. |
| `asociacion_multicamara.py` | `AsociadorMulticamara`: galería compartida, gating físico, fusiones y reportes (`global_id`). |
| `motor.py` | `MotorLAP01`, vista en vivo, `ResultadoMulticamara`, `LectorSincronizado`, `procesar_videos`, `armar_resultado`. |
| `mapa_2d.py` | `EstabilizadorCamaras`, `ModeloPiso`, `CalibradorMapa`, `proyectar_en_mapa`, `ReagrupadorMapa`, `DibujoMapa`. |

| Carpeta | Contenido |
|---|---|
| `../dataset/` | Videos de las tres cámaras (compartidos con Test Modelo). |
| `Modelo/` | Pesos y configuración de origen: `yolo26m.pt`, `resnet18-f37072fd.pth`, `yolo26s-reid.onnx`, `clip_genero/`, `config_lap01.json`, `camaras.json`, `mapa_piso.json`. |
| `Ouput/` | Resultados del build: videos procesados, `mapa_2d.mp4`, `mapa_trayectorias.png`, `trajectory_points.csv`, `identidades_genero.csv`. |
| `../Test Modelo/Modelo/` | **El modelo terminado** (sección 9): pesos, configuración, mapa y una copia de `lap01/`. Lo usa `Test Modelo/Test_Tiempo_Real.ipynb`. |

Si cambias un `.py`, reinicia el kernel (o usa `importlib.reload`) y vuelve a ejecutar. No se entrena ni se descarga nada: todos los pesos se leen de `Modelo/`.

## 1. Entorno y código

Ultralytics queda sin descargas, autoinstalación ni telemetría, y guarda su configuración en `Modelo/.runtime/`. Las variables se fijan antes de importar `lap01`.

In [ ]:
from pathlib import Path
import os
import sys

BUILD = next((carpeta.resolve() for carpeta in (Path.cwd(), Path.cwd() / "Build Modelo")
              if (carpeta / "lap01" / "__init__.py").is_file() and (carpeta / "Modelo" / "config_lap01.json").is_file()),
             None)
if BUILD is None:
    raise FileNotFoundError("Abre Jupyter desde la carpeta Build Modelo.")
BASE = BUILD.parent
DATASET = BASE / "dataset"
MODELO = BUILD / "Modelo"
CODIGO = BUILD / "lap01"
OUTPUT = BUILD / "Ouput"
OUTPUT.mkdir(exist_ok=True)
TEST_MODELO = BASE / "Test Modelo" / "Modelo"
NOTEBOOK = BUILD / "Build_Modelo.ipynb"

if "ultralytics" in sys.modules or "lap01" in sys.modules:
    print("Ultralytics o lap01 ya estaban importados: reinicia el kernel para aplicar la configuración y los cambios.")
os.environ["YOLO_CONFIG_DIR"] = str(MODELO / ".runtime" / "ultralytics")
os.environ["YOLO_OFFLINE"] = "1"
os.environ["YOLO_AUTOINSTALL"] = "0"
sys.path.insert(0, str(BUILD))

from pathlib import Path
import hashlib
import heapq
import importlib.metadata
import json
import math
import time
import uuid
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
from contextlib import ExitStack
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import datetime
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from scipy.optimize import linear_sum_assignment
from torch import nn
from threadpoolctl import threadpool_limits
import shutil

import lap01
from lap01 import *
from lap01.tracking_local import _iou

print("Python:", sys.executable)
print("Carpeta:", BUILD)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible (CPU)")

## 2. Configurar el build

Cada archivo `dataset/cam0X_*.mp4` es una cámara distinta; sus identificadores deben coincidir con `Modelo/camaras.json`. **Tiempo común = tiempo del video + `timestamp_offset`** (en `camaras.json`).

| Parámetro | Qué hace |
|---|---|
| `MAX_FRAMES` | `None` procesa los videos completos; `150` = prueba de 10 s. No se saltan frames. |
| `INICIOS` | Frame inicial de cada cámara (recorta sin cambiar el reloj). |
| `TIEMPO_REAL` | `True` frena al ritmo del video (demo); `False` va lo más rápido posible. Mismo resultado. |
| `VISTA_EN_VIVO`, `VISTA_FPS`, `VISTA_ANCHO` | Mosaico de las cámaras mientras se procesa. |
| `DETECTOR` | `"yolo26m.pt"` (recomendado: tiempo real a 15 FPS con IDs más estables) o `"yolo26s.pt"` (−44 % de GPU para más cámaras o equipos modestos; detecta menos personas lejanas y los IDs son menos estables). El build publica el elegido al Test. |
| `DEVICE`, `BATCH_YOLO`, `CPU_THREADS` | `auto` usa la GPU si hay; un lote YOLO con un frame por cámara; hilos de CPU. |
| `INICIO_GRABACION` | Hora real de la grabación para el `TIMESTAMPTZ` de `trajectory_points` (p. ej. `"2026-09-12T14:30:00-05:00"`); `None` = ahora. |
| `MAPA_2D`, `RECALIBRAR_MAPA` | Ubica a cada persona en metros; recalibra aunque exista `Modelo/mapa_piso.json`. |
| `UNIR_CON_MAPA`, `TEMA_MAPA` | Reagrupa identidades con el mapa; tema `"claro"` u `"oscuro"`. |
| `EXPORTAR_*` | Videos por cámara, mapa (`mapa_2d.mp4` + PNG), `trajectory_points.csv` y `identidades_genero.csv` en `Ouput/`. |
| `PUBLICAR_EN_TEST` | Copia el modelo terminado a `../Test Modelo/Modelo/`. |
| `PUBLICAR_EN_BD`, `URL_WEB`, `NOMBRE_SESION` | Guarda la sesión (personas, tracklets, trayectorias, mapa y cámaras) en la base de la web, que la muestra en En vivo e Insights → ESAN. |

In [ ]:
VIDEOS = {ruta.name.split("_")[0]: ruta for ruta in sorted(DATASET.glob("cam*.mp4"))}

MAX_FRAMES = None
INICIOS = {cid: 0 for cid in VIDEOS}
TIEMPO_REAL = False
VISTA_EN_VIVO = True
VISTA_FPS = 8.0
VISTA_ANCHO = 960
DETECTOR = "yolo26m.pt"
DEVICE = "auto"
BATCH_YOLO = True
CPU_THREADS = 2
INICIO_GRABACION = None

MAPA_2D = True
RECALIBRAR_MAPA = False
UNIR_CON_MAPA = True
TEMA_MAPA = "claro"

EXPORTAR_VIDEOS = True
EXPORTAR_MAPA = True
EXPORTAR_TRAYECTORIAS = True
EXPORTAR_GENEROS = True
PUBLICAR_EN_TEST = True
PUBLICAR_EN_BD = True
URL_WEB = "http://localhost:8080"
NOMBRE_SESION = "ESAN · dataset 3 cámaras"

CONFIG = json.loads((MODELO / "config_lap01.json").read_text())
CONFIG["weights"]["detector"] = DETECTOR
CONFIG_CAMARAS = json.loads((MODELO / "camaras.json").read_text())
if set(VIDEOS) != set(CONFIG_CAMARAS["cameras"]):
    raise ValueError(f"Cámaras del dataset {sorted(VIDEOS)} ≠ Modelo/camaras.json {sorted(CONFIG_CAMARAS['cameras'])}.")

### Videos de entrada

Se leen los metadatos de cada video sin cargarlo en memoria. `area` es la zona con imagen real de cada cámara: las bandas negras del recorte no cuentan como borde de la escena.

In [ ]:
display(pd.DataFrame(inspeccionar_videos(VIDEOS)))
print("Modo multicámara:", CONFIG_CAMARAS["mode"])
display(pd.DataFrame({cid: {"timestamp_offset": cam.get("timestamp_offset", 0.0),
                            "resolution": cam.get("resolution"),
                            "homografia": cam.get("H") is not None}
                      for cid, cam in CONFIG_CAMARAS["cameras"].items()}).T)

## 3. Cargar los modelos

Se cargan `yolo26m.pt` (detección), `resnet18-f37072fd.pth` (apariencia local), `yolo26s-reid.onnx` (Re-ID entre cámaras) y, si `gender.enabled = true`, el CLIP local `clip_genero/`. El Re-ID y CLIP usan la GPU de `jupyterev` cuando está disponible; si el Re-ID no puede convertirse, usa ONNX Runtime en CPU.

Para usar solo la ResNet18 en la asociación, antes de esta celda cambia `CONFIG["multicamera_encoder"] = {"type": "resnet18"}`. Cada encoder necesita validar sus umbrales con ejemplos etiquetados.

In [ ]:
motor = MotorLAP01(MODELO, CONFIG, device=DEVICE, batch=BATCH_YOLO)
asociador = crear_asociador(motor, CONFIG_CAMARAS)

print("Detector YOLO26m y ResNet18:", motor.device)
print("Encoder multicámara:", motor.config["multicamera_encoder"]["weights"],
      "·", getattr(asociador.reid, "backend", "ResNet18"))
if motor.genero.enabled:
    print("Género estimado (cuerpo, sin rostro):", motor.genero.weights.name, "·", motor.genero.device)
print("Modo multicámara:", asociador.mode)
display(pd.DataFrame({"tracker": pd.Series(motor.config["tracker"]),
                      "association": pd.Series(asociador.config.get("association", {})),
                      "gender": pd.Series(motor.config.get("gender", {}))}))

## 4. Procesar las tres cámaras

El mosaico muestra `cam01`, `cam02` y `cam03` en vertical. `G7 | cam02 L3 | H` indica identidad global 7, track local 3 y la estimación corporal estable de género (`H`/`M`); si no hay evidencia suficiente, se omite el último campo. Los GID en vivo son decisiones *online* y pueden cambiar si más adelante aparece evidencia de fusión o un conflicto. El botón ■ interrumpe el procesamiento y conserva lo ya procesado.

In [ ]:
vista = VistaEnVivo(VISTA_ANCHO, camaras=len(VIDEOS)) if VISTA_EN_VIVO else None
resultado = procesar_videos(
    motor, asociador, VIDEOS,
    max_frames=MAX_FRAMES,
    tiempo_real=TIEMPO_REAL,
    preview=vista,
    preview_fps=VISTA_FPS,
    preview_width=VISTA_ANCHO,
    start_frames=INICIOS,
    cpu_threads=CPU_THREADS,
    inicio_grabacion=INICIO_GRABACION,
)
if vista is not None:
    vista.terminar(resultado)

r = resultado.resumen
print(f"Sesión {r['session_id']} · {r['status']} · {r['frames_total']} frames · {r['video_seconds']:.1f} s de video")
print(f"Capacidad: {r['fps_per_camera']:.2f} FPS por cámara ({r['fps_total']:.2f} FPS totales) en {r['device']}")

## 5. Auditoría

### 8.1 — Seguimiento local por cámara (LAP01)

Mide la fragmentación de IDs (IDs de 1–2 frames y duración mediana), las reapariciones (el mismo `local_id` vuelve tras frames ausente), los solapes entre cajas (IoU ≥ 0.12, es decir, cruces) y los diagnósticos internos del tracker. **No es ground truth:** IDF1 y los cambios de ID requieren anotaciones de referencia.

In [ ]:
observaciones = resultado.observaciones()

auditoria = []
for cid in VIDEOS:
    grupo = observaciones[observaciones.camera_id == cid]
    fila = {"camera_id": cid, **resultado.resumen["cameras"][cid]}
    if not grupo.empty:
        duracion = grupo.groupby("local_id").frame.nunique()
        huecos = np.concatenate([np.diff(np.sort(g.frame.unique())) - 1 for _, g in grupo.groupby("local_id")])
        huecos = huecos[huecos > 0]
        solapes = 0
        for _, cuadro in grupo.groupby("frame"):
            cajas = cuadro[["x1", "y1", "x2", "y2"]].to_numpy()
            solapes += sum(_iou(cajas[i], cajas[j]) >= 0.12 for i in range(len(cajas)) for j in range(i + 1, len(cajas)))
        fila.update(ids_locales=int(grupo.local_id.nunique()), ids_cortos_1_2_frames=int((duracion <= 2).sum()),
                    duracion_mediana_frames=float(duracion.median()), reapariciones=int(len(huecos)),
                    frames_ocultos_max=int(huecos.max()) if len(huecos) else 0, solapes_iou_012=int(solapes))
    auditoria.append(fila)
display(pd.DataFrame(auditoria).set_index("camera_id").T)
if motor.genero.enabled:
    print("Estimación de género por identidad global (CLIP corporal; Sin determinar = evidencia insuficiente):")
    display(resultado.tabla_generos())

### 8.2 — Galería compartida e identidades multicámara (antes del mapa)

`galeria` muestra lo que las tres cámaras saben de cada persona contada: vistas por cámara y por orientación (frente, espalda, costado, quieto), segundos visible y última cámara donde se vio. `identidades` agrupa los tracklets de cada `global_id`, `tracklets` muestra su intervalo y sus muestras Re-ID, y `enlaces` registra cada fusión (`associate`) y separación (`separate`) con su puntaje. Los puntajes son similitudes, no probabilidades de acierto.

Son las decisiones del Re-ID. La sección 9 las revisa con el mapa y reconstruye estas tablas.

In [ ]:
display(pd.Series(resultado.resumen["multicamera"], name="multicámara"))
print("Galería global compartida (lo que todas las cámaras saben de cada persona contada):")
display(pd.DataFrame(resultado.galeria))

identidades = pd.DataFrame(resultado.identidades)
if not identidades.empty:
    identidades["n_camaras"] = identidades.camaras.map(len)
    identidades = identidades.sort_values(["n_camaras", "global_id"], ascending=[False, True])
display(identidades.head(30))
tracklets = pd.DataFrame(resultado.tracklets)
if not tracklets.empty:
    tracklets = tracklets.sort_values(["global_id", "inicio_s"])
display(tracklets.head(30))
display(pd.DataFrame(resultado.enlaces).tail(30))

## 6. Mapa 2D (`lap01/mapa_2d.py`)

Estabiliza, carga (o calibra) el mapa y ubica cada detección en metros.

In [ ]:
if not MAPA_2D:
    print("Mapa 2D desactivado (MAPA_2D = False).")
else:
    estabilizacion = EstabilizadorCamaras()(resultado)
    print("Estabilización:", {cid: f"{len(H)} frames" for cid, H in estabilizacion.items()})

    mapa = None if RECALIBRAR_MAPA else cargar_mapa(MODELO)
    if mapa is None:
        mapa = CalibradorMapa().calibrar(resultado, estabilizacion, CONFIG_CAMARAS)
        guardar_mapa(mapa, MODELO)
        print("Mapa calibrado y guardado en Modelo/mapa_piso.json")
    else:
        print("Mapa cargado de Modelo/mapa_piso.json (RECALIBRAR_MAPA = True para volver a calibrar)")

    proyectar_en_mapa(resultado, estabilizacion, mapa)
    display(pd.DataFrame(mapa["informe"]).T)
    print(f"Piso: {mapa['tam_px'][0] / mapa['px_por_metro']:.1f} m × {mapa['tam_px'][1] / mapa['px_por_metro']:.1f} m · "
          f"desfases {mapa['desfases_s']}")

Reagrupa las identidades con el mapa y muestra qué cambió respecto del Re-ID (`antes` usa los IDs de la sección 8.2).

In [ ]:
if MAPA_2D and UNIR_CON_MAPA:
    cambios = ReagrupadorMapa(min_muestras=asociador.min_samples, min_visible_s=asociador.min_duration_s)(resultado)
    resultado.actualizar_generos(motor.genero.consolidar_globales(resultado.filas))
    multi = resultado.resumen["multicamera"]
    print(f"Con el mapa: {multi['uniones_mapa']} personas unidas y {multi['separados_mapa']} tracklets separados del Re-ID · "
          f"personas contadas {multi['identidades_globales']} · multicámara {multi['identidades_multicamara']}")
    display(pd.DataFrame(cambios))
    display(pd.DataFrame(resultado.galeria))

Vista de las trayectorias en el mapa (la misma imagen que se exporta en la sección 11).

In [ ]:
if MAPA_2D:
    display(Image(data=cv2.imencode(".png", DibujoMapa(resultado, mapa, TEMA_MAPA).trayectorias())[1].tobytes(), width=900))

## 7. Tabla `trajectory_points`

Cada fila indica dónde estuvo un `global_id` en un instante, después del Re-ID multicámara y de la reagrupación con el mapa. Por ahora **no hay zonas**, así que `zone_id` queda `NULL`. Tampoco se generan `zones` ni `spatial_events`.

Las **pasadas breves** que no se unieron a ninguna persona no se cuentan y no se exportan.

| Campo | Origen en este notebook |
|---|---|
| `point_id` | `BIGSERIAL`: lo asigna PostgreSQL (no va en el CSV). |
| `global_id` | UUID v5 de (sesión, GID final tras la reagrupación con el mapa): identidad anónima de sesión. |
| `timestamp` | `INICIO_GRABACION` + tiempo del video + desfase medido de la cámara (sincronizado). |
| `camera_id`, `local_id` | Trazabilidad hacia la cámara y el tracker local. |
| `tracklet_id` | UUID v5 de (sesión, `cam/L/T`). |
| `x`, `y`, `geom` | Pies de la persona en el mapa 2D del piso (metros, sección 9); `NULL` si los pies están cortados por el borde o sin mapa. |
| `speed_mps`, `direction_deg` | Movimiento del tracklet en el mapa (ventana de 1 s); dirección en [0, 360) respecto del eje x del mapa. |
| `confidence` | Confianza de YOLO26m para esa detección. |

Como las cámaras se solapan, un mismo `global_id` puede tener varias filas en el mismo instante (una por cámara). La consolidación de duplicados pertenece a la Parte III.

```sql
CREATE TABLE trajectory_points (
    point_id      BIGSERIAL PRIMARY KEY,
    global_id     UUID             NOT NULL,
    "timestamp"   TIMESTAMPTZ      NOT NULL,
    camera_id     VARCHAR(30)      NOT NULL,
    local_id      BIGINT           NOT NULL,
    tracklet_id   UUID             NOT NULL,
    x             DOUBLE PRECISION,
    y             DOUBLE PRECISION,
    geom          geometry(Point),
    zone_id       VARCHAR(50),          -- FK a zones(zone_id) cuando exista la tabla zones
    speed_mps     REAL,
    direction_deg REAL,
    confidence    REAL             NOT NULL
);

\copy trajectory_points (global_id, "timestamp", camera_id, local_id, tracklet_id, x, y, geom, zone_id, speed_mps, direction_deg, confidence) FROM 'Build Modelo/Ouput/trajectory_points.csv' WITH (FORMAT csv, HEADER true)
```

In [ ]:
trajectory_points = resultado.trajectory_points()

assert list(trajectory_points.columns) == COLUMNAS_TRAJECTORY_POINTS
assert trajectory_points.zone_id.isna().all()
assert trajectory_points[["global_id", "timestamp", "camera_id", "local_id", "tracklet_id", "confidence"]].notna().all().all()
if not trajectory_points.empty:
    assert trajectory_points.camera_id.str.len().max() <= 30
    assert all(uuid.UUID(v) for v in trajectory_points.global_id.unique())

con_plano = int(trajectory_points.x.notna().sum())
print(f"Filas: {len(trajectory_points)} · global_id: {trajectory_points.global_id.nunique()} · "
      f"tracklets: {trajectory_points.tracklet_id.nunique()} · con x/y: {con_plano}")
if not con_plano:
    print("Sin mapa: x, y, geom, speed_mps y direction_deg quedan NULL. Activa MAPA_2D (sección 9).")
display(trajectory_points.dtypes.to_frame("dtype").T)
display(trajectory_points.head(20))

## 8. Exportar

Solo se exporta lo que se va a usar:

- **`Build Modelo/Ouput/`**: `cam01_procesado.mp4`, `cam02_procesado.mp4` y `cam03_procesado.mp4` (con los `global_id` finales), `mapa_2d.mp4` (mapa del piso con cada persona en cada instante, junto a las cámaras sincronizadas), `mapa_trayectorias.png`, `trajectory_points.csv` e `identidades_genero.csv` (una estimación consolidada por identidad). Cada ejecución reemplaza los archivos anteriores. No se guardan recortes, embeddings ni frames sueltos.
- **`Modelo/`**: `config_lap01.json` (la configuración usada) y `exportacion.json` (manifiesto SHA-256 de pesos, configuración y notebook, más las versiones de paquetes). `mapa_piso.json` guarda la calibración del mapa (sección 9).

Los tres videos por cámara se escriben a la vez (~40 s en total con el mapa). La exportación del modelo (segunda celda) no depende de haber procesado videos.

In [ ]:
if EXPORTAR_VIDEOS:
    for cid, ruta in resultado.exportar_videos(OUTPUT, motor.config["detector"]["display_conf"]).items():
        print("Video procesado:", ruta.relative_to(BASE))
if EXPORTAR_MAPA and MAPA_2D:
    imagen, video = DibujoMapa(resultado, mapa, TEMA_MAPA).exportar(OUTPUT)
    print("Mapa 2D:", imagen.relative_to(BASE), "·", video.relative_to(BASE))
if EXPORTAR_TRAYECTORIAS:
    print("Tabla:", resultado.guardar_trajectory_points(OUTPUT / "trajectory_points.csv").relative_to(BASE))
if EXPORTAR_GENEROS and motor.genero.enabled:
    print("Géneros:", resultado.guardar_generos(OUTPUT / "identidades_genero.csv").relative_to(BASE))

In [ ]:
def _sha256(ruta):
    with Path(ruta).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def exportar_modelo(config, config_camaras):
    """Guarda la configuración usada y un manifiesto verificable en Modelo/. No copia ni modifica los pesos."""
    cargar_camaras(config_camaras)
    pesos = set(config["weights"].values())
    encoder = config.get("multicamera_encoder", {})
    if encoder.get("type") == "onnx":
        pesos.add(encoder["weights"])
    genero = config.get("gender", {})
    archivos_genero = []
    if genero.get("enabled", False):
        nombre = genero.get("weights", "")
        if not isinstance(nombre, str) or Path(nombre).name != nombre:
            raise ValueError("gender.weights debe ser una carpeta local dentro de Modelo/.")
        carpeta_genero = MODELO / nombre
        if not (carpeta_genero / "config.json").is_file() or not (carpeta_genero / "model.safetensors").is_file():
            raise FileNotFoundError(f"Falta el modelo CLIP local en {carpeta_genero}")
        archivos_genero = sorted(ruta for ruta in carpeta_genero.rglob("*") if ruta.is_file())
    faltantes = [nombre for nombre in sorted(pesos) if Path(nombre).name != nombre or not (MODELO / nombre).is_file()]
    if faltantes:
        raise FileNotFoundError(f"Faltan pesos locales en Modelo/: {faltantes}")

    ruta_config = MODELO / "config_lap01.json"
    ruta_config.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
    archivos = {f"Modelo/{nombre}": _sha256(MODELO / nombre)
                for nombre in [*sorted(pesos), "config_lap01.json", "camaras.json", "mapa_piso.json"]
                if (MODELO / nombre).is_file()}
    archivos.update({f"Modelo/{ruta.relative_to(MODELO)}": _sha256(ruta) for ruta in archivos_genero})
    versiones = {}
    for paquete in ("torch", "torchvision", "transformers", "ultralytics", "numpy", "scipy", "opencv-python", "onnxruntime", "pandas"):
        try:
            versiones[paquete] = importlib.metadata.version(paquete)
        except importlib.metadata.PackageNotFoundError:
            versiones[paquete] = None
    manifiesto = {"version": config.get("version", "2.0"),
                  "fecha": datetime.now().astimezone().isoformat(timespec="seconds"),
                  "notebook": NOTEBOOK.name, "notebook_sha256": _sha256(NOTEBOOK) if NOTEBOOK.is_file() else None,
                  "codigo_sha256": {f"lap01/{ruta.name}": _sha256(ruta) for ruta in sorted(CODIGO.glob("*.py"))},
                  "modo_multicamara": config_camaras.get("mode"), "files_sha256": archivos, "packages": versiones}
    ruta_manifiesto = MODELO / "exportacion.json"
    ruta_manifiesto.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2) + "\n")
    return ruta_config, ruta_manifiesto


for ruta in exportar_modelo(CONFIG, CONFIG_CAMARAS):
    print("Modelo exportado:", ruta.relative_to(BASE))

## 9. Publicar el modelo en `Test Modelo/Modelo/`

Se copia solo lo que hace falta para usar el modelo, sin los resultados del build:

- **Pesos:** `yolo26m.pt`, `resnet18-f37072fd.pth`, `yolo26s-reid.onnx` y `clip_genero/` (si `gender.enabled`). Solo se copian si cambiaron.
- **Configuración:** `config_lap01.json`, `camaras.json` y `exportacion.json` (manifiesto SHA-256).
- **Mapa:** `mapa_piso.json`, más `piso_L_m`, la silueta del piso que dibuja el mapa en vivo.
- **Código:** una copia de `lap01/`. Si cambias un `.py`, vuelve a ejecutar esta sección para que el test use la versión nueva.

In [ ]:
def _copiar(origen, destino):
    """Copia solo si cambió (tamaño o fecha): los pesos de CLIP pesan ~580 MB."""
    if destino.is_file() and destino.stat().st_size == origen.stat().st_size \
            and int(destino.stat().st_mtime) == int(origen.stat().st_mtime):
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(origen, destino)


def publicar_en_test(config, destino=TEST_MODELO, piso_L_m=None):
    """Guarda en Test Modelo/Modelo/ todo lo necesario para usar el modelo: pesos, configuración, mapa y código lap01/."""
    pesos = set(config["weights"].values())
    if config.get("multicamera_encoder", {}).get("type") == "onnx":
        pesos.add(config["multicamera_encoder"]["weights"])
    archivos = [MODELO / nombre for nombre in sorted(pesos)]
    genero = config.get("gender", {})
    if genero.get("enabled", False):
        archivos += sorted(ruta for ruta in (MODELO / genero["weights"]).rglob("*") if ruta.is_file())
    archivos += [MODELO / nombre for nombre in ("config_lap01.json", "camaras.json", "exportacion.json")]
    for ruta in archivos:
        _copiar(ruta, destino / ruta.relative_to(MODELO))
    mapa_test = cargar_mapa(MODELO)
    if mapa_test is not None:
        if piso_L_m is not None:
            mapa_test["piso_L_m"] = piso_L_m
        guardar_mapa(mapa_test, destino)
    codigo_test = destino / "lap01"
    for viejo in codigo_test.glob("*.py"):
        if not (CODIGO / viejo.name).is_file():
            viejo.unlink()
    shutil.copytree(CODIGO, codigo_test, ignore=shutil.ignore_patterns("__pycache__"), dirs_exist_ok=True)
    return destino


if PUBLICAR_EN_TEST:
    piso = DibujoMapa(resultado, mapa, TEMA_MAPA)._piso_L().tolist() if MAPA_2D else None
    destino = publicar_en_test(CONFIG, piso_L_m=piso)
    print("Modelo publicado en:", destino.relative_to(BASE))
    for ruta in sorted(destino.iterdir()):
        print("  ", ruta.name + ("/" if ruta.is_dir() else ""))

## 10. Publicar en la base de datos

Guarda la sesión en PostgreSQL a través de la web (`POST /api/v1/esan/sessions`): identidades con su género, tracklets y cada punto de `trajectory_points`, además del mapa del piso y las cámaras. La web la muestra en **En vivo → ESAN** (reproducción sobre el plano con los videos sincronizados) e **Insights → ESAN**. Volver a publicar la misma sesión la reemplaza. Requiere la web levantada (`docker compose up -d`).

In [ ]:
if PUBLICAR_EN_BD:
    piso = DibujoMapa(resultado, mapa, TEMA_MAPA)._piso_L().tolist() if MAPA_2D else None
    carga = sesion_para_bd(resultado, nombre=NOMBRE_SESION, kind="BUILD", mapa={**mapa, "piso_L_m": piso} if MAPA_2D else None,
                           config_camaras=CONFIG_CAMARAS, config=CONFIG)
    inicio = time.perf_counter()
    try:
        respuesta = publicar_sesion(carga, URL_WEB)
        print(f"Sesión {respuesta['session_id']} guardada: {respuesta['identities']} personas, {respuesta['tracklets']} tracklets, "
              f"{respuesta['points']} puntos · {respuesta['ms']} ms en la base, {time.perf_counter() - inicio:.1f} s en total")
    except OSError as error:
        print(f"La web no responde en {URL_WEB} ({error}). Levántala con: docker compose up -d")

## 11. Calibración manual con medidas reales (cámaras fijas)

Esta sección es para **cámaras fijas** con puntos medidos en el lugar. Estos videos tienen cámaras que se mueven, así que para ellos se usa el mapa automático de la sección 9. Si más adelante tienes medidas reales, puedes usarlas aquí.

En cada cámara marca **al menos cuatro puntos del suelo** sobre el video de 1280×720 y sus coordenadas en un plano común en metros. Reserva **al menos dos puntos distintos** para validar. No uses esquinas de paredes ni objetos a otra altura. Si cambia el recorte o la resolución, vuelve a calibrar.

Completa `CALIBRACIONES` para las tres cámaras. La función rechaza puntos degenerados, controles repetidos y errores mayores que el límite. Revisa también en `camaras.json` los polígonos de cobertura, los solapes reales y las transiciones con sus tiempos. Al guardar, el modo pasa a `calibrado`. Luego vuelve a ejecutar el notebook desde el principio.

Formato por cámara (píxeles del video ↔ metros del plano):

```python
CALIBRACIONES["cam01"] = {
    "imagen": [[u, v], ...],         "plano": [[X, Y], ...],           # ≥ 4 puntos de ajuste
    "control_imagen": [[u, v], ...], "control_plano": [[X, Y], ...],   # ≥ 2 puntos de validación
}
```

In [ ]:
CALIBRACIONES = {}
if CALIBRACIONES:
    if set(CALIBRACIONES) != set(VIDEOS):
        raise ValueError("Completa todas las cámaras antes de activar el modo calibrado.")
    calibrada = deepcopy(CONFIG_CAMARAS)
    for cid, puntos in CALIBRACIONES.items():
        limite = float(calibrada["cameras"][cid].get("max_error_m", 0.75))
        calibrada["cameras"][cid].update(calibrar_homografia(**puntos, max_error_m=limite))
    calibrada["mode"] = "calibrado"
    cargar_camaras(calibrada, VIDEOS)
    (MODELO / "camaras.json").write_text(json.dumps(calibrada, indent=2, ensure_ascii=False) + "\n")
    CONFIG_CAMARAS = calibrada
    display(pd.DataFrame({cid: cam["validation"] for cid, cam in calibrada["cameras"].items()}).T)
    print("Calibración validada y guardada en Modelo/camaras.json. Vuelve a ejecutar el notebook.")